In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

In [0]:
df_dp_silver = spark.table("projeto_seguranca_rj.bronze.dp_municipio")

In [0]:
df_dp_silver.printSchema()

In [0]:
df_dp_silver.columns

In [0]:
colunas_codigo = [
    "cisp",
    "aisp",
    "risp",
    "mcirc"
]

for coluna in colunas_codigo:
    df_dp_silver = (
        df_dp_silver
        .withColumn(
            coluna,
            F.col(coluna).cast("string")
        )
    )

In [0]:
df_dp_silver.select(
    [
        F.count(
            F.when(
                F.col(c).isNull(),
                c
            )
        ).alias(c)
        for c in df_dp_silver.columns
    ]
).show()

In [0]:
colunas_nulas_dp = [
    "feminicidio",
    "tentativa_feminicidio",
    "roubo_cx_eletronico",
    "roubo_bicicleta",
    "furto_bicicleta",
    "posse_drogas",
    "trafico_drogas",
    "apreensao_drogas_sem_autor",
    "apf",
    "aaapai",
    "cmp",
    "cmba"
]

for coluna in colunas_nulas_dp:
    df_dp_silver = (
        df_dp_silver
        .withColumn(
            coluna,
            F.coalesce(
                F.col(coluna),
                F.lit(0)
            )
        )
    )

In [0]:
df_dp_silver.printSchema()

In [0]:
# Padronização da região Grande Niterói
# Corrige tanto a grafia normal quanto possíveis versões corrompidas por encoding.
df_dp_silver = (
    df_dp_silver
    .withColumn(
        "regiao",
        F.when(
            F.col("regiao").contains("Niter"),
            F.lit("Grande Niterói")
        ).otherwise(F.col("regiao"))
    )
)


In [0]:
# Conferência dos valores de região após a padronização
display(
    df_dp_silver
    .select("regiao")
    .distinct()
    .orderBy("regiao")
)


In [0]:
# Diagnóstico: chaves repetidas antes da deduplicação
display(
    df_dp_silver
    .groupBy(
        "cisp",
        "munic",
        "ano",
        "mes"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)


In [0]:
# 1) Remove registros totalmente idênticos
df_dp_silver = df_dp_silver.dropDuplicates()

# 2) Para a mesma chave temporal, mantém a maior fase
# fase 3 = consolidado com errata/correção
# fase 2 = consolidado sem errata
w_fase = (
    Window
    .partitionBy(
        "cisp",
        "munic",
        "ano",
        "mes"
    )
    .orderBy(
        F.col("fase").desc()
    )
)

df_dp_silver = (
    df_dp_silver
    .withColumn(
        "rn",
        F.row_number().over(w_fase)
    )
    .filter(
        F.col("rn") == 1
    )
    .drop("rn")
)


In [0]:
# Validação final: deve haver uma única linha por CISP/município/ano/mês
duplicados_finais = (
    df_dp_silver
    .groupBy(
        "cisp",
        "munic",
        "ano",
        "mes"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

qtd_duplicados = duplicados_finais.count()

print(
    "Chaves duplicadas restantes:",
    qtd_duplicados
)

if qtd_duplicados > 0:
    display(duplicados_finais)
    raise ValueError(
        "Ainda existem duplicidades em dp_municipio. "
        "A tabela Silver não será salva."
    )
else:
    print("Validação de unicidade concluída com sucesso.")


In [0]:
# Validação da padronização de Grande Niterói
regioes_niteroi = (
    df_dp_silver
    .filter(
        F.col("regiao").contains("Niter")
    )
    .select("regiao")
    .distinct()
)

display(regioes_niteroi)

valores_niteroi = [
    linha["regiao"]
    for linha in regioes_niteroi.collect()
]

if valores_niteroi and valores_niteroi != ["Grande Niterói"]:
    raise ValueError(
        "Ainda existem valores inconsistentes para Grande Niterói."
    )


In [0]:
# Persistência da Silver já limpa e validada
(
    df_dp_silver.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        "projeto_seguranca_rj.silver.dp_municipio"
    )
)

print(
    "Tabela projeto_seguranca_rj.silver.dp_municipio "
    "salva com sucesso."
)
